[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke08-etikk-implementering/05_trustworthy_ai_i_helse.ipynb)

# Trustworthy AI i helse

> **Målet med ansvarlig AI er ikke bare høy ytelse, men systemer som er robuste, forståelige, overvåkbare og trygge å bruke i praksis.**

I denne notebooken ser vi på hva som må være på plass før et AI-system kan være verdig tillit i helsetjenesten. Det betyr at vi må tenke utover presisjon og modellscore, og også se på usikkerhet, distribusjonsskifte, menneskelig kontroll, validering og monitorering.

## 🎯 Hva du lærer i dag

✅ forklare hva trustworthy AI betyr i helsekontekst  
✅ forstå robusthet, usikkerhet og distribusjonsskifte  
✅ beskrive hvorfor human-in-the-loop er sentralt i medisinsk AI  
✅ forklare behovet for validering før innføring og monitorering etterpå  

## 📚 Innhold

1. Hva menes med trustworthy AI?  
2. Sentrale prinsipper  
3. Robusthet og distribusjonsskifte  
4. Usikkerhet og når AI bør si «jeg vet ikke»  
5. Human-in-the-loop  
6. Validering og kontinuerlig monitorering  
7. Oppsummering

### Men først: 🔧 miljøoppsett – kode skal fungere både lokalt og i Google Colab

In [ ]:
import sys
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

warnings.filterwarnings("ignore")
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    !pip install seaborn --quiet
    if not os.path.exists('AI-og-helse'):
        try:
            !git clone https://github.com/arvidl/AI-og-helse.git
            print("✅ Repository klonet")
        except Exception:
            print("⚠️ Kunne ikke klone repository automatisk")
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Arbeidsmappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokalt miljø")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("✅ Miljøet er konfigurert")
print(f"🎲 Random seed satt til {RANDOM_SEED}")

## 1. Hva menes med trustworthy AI?

Et AI-system i helse er ikke tillitsverdig bare fordi det scorer høyt på én test. Det må også være:

- **robust** når data eller situasjoner endrer seg
- **transparent** nok til å kunne brukes forsvarlig
- **rettferdig** på tvers av pasientgrupper
- **sikkert** når modellen er usikker eller tar feil
- **overvåkbar** etter at systemet er tatt i bruk

Denne notebooken går mindre inn i juss og etiske prinsipper i seg selv, og mer inn i hva som må være på plass for at et system skal være forsvarlig i faktisk bruk. Den bygger derfor direkte videre på:

- `04_ai_etikk_i_medisinen.ipynb`
- `01_gdpr_personvern.ipynb`
- `02_bias_rettferdighet.ipynb`
- `03_ce_mdr_regulering.ipynb`

I praksis betyr dette at tillit til AI må **fortjenes gjennom dokumentasjon, validering og oppfølging**.

In [ ]:
def vurder_tillitsverdighet():
    kriterier = {
        "Menneskelig kontroll": "Kan en kliniker forstå, vurdere og overstyre systemet?",
        "Robusthet": "Er systemet testet på realistiske og krevende situasjoner?",
        "Personvern": "Er data behandlet på en lovlig og forsvarlig måte?",
        "Transparens": "Er begrensninger, datagrunnlag og ytelse dokumentert?",
        "Rettferdighet": "Fungerer systemet rimelig godt for ulike pasientgrupper?",
        "Monitorering": "Finnes det plan for drift, overvåking og oppdatering?"
    }

    print("SJEKKLISTE FOR TRUSTWORTHY AI I HELSE")
    print("=" * 55)
    for navn, spørsmål in kriterier.items():
        print(f"☐ {navn}")
        print(f"   {spørsmål}")
        print()

vurder_tillitsverdighet()

## 2. Robusthet og distribusjonsskifte

Et vanlig problem i medisinsk AI er at modellen er trent på én type data, men senere brukes på en annen pasientgruppe, et annet sykehus eller en annen arbeidsflyt.

Dette kalles **distribusjonsskifte**. En modell som ser god ut under utvikling, kan derfor bli mindre pålitelig når den møter virkeligheten.

Eksempel:

- treningsdata kommer fra ett universitetssykehus
- produksjonsdata kommer fra en geriatrisk avdeling med eldre pasienter
- modellen møter nå et annet mønster enn det den lærte på

In [ ]:
# Demonstrasjon av distribusjonsskifte

training_age = np.random.normal(55, 15, 1000)
training_age = training_age[(training_age > 20) & (training_age < 90)]

production_age = np.random.normal(75, 10, 500)
production_age = production_age[(production_age > 50) & (production_age < 95)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(training_age, bins=30, alpha=0.6, label='Treningsdata', color='steelblue', density=True, edgecolor='black', linewidth=0.5)
ax.hist(production_age, bins=30, alpha=0.6, label='Produksjonsdata', color='indianred', density=True, edgecolor='black', linewidth=0.5)
ax.axvline(training_age.mean(), color='steelblue', linestyle='--', linewidth=2, label=f'Treningssnitt: {training_age.mean():.1f} år')
ax.axvline(production_age.mean(), color='indianred', linestyle='--', linewidth=2, label=f'Produksjonssnitt: {production_age.mean():.1f} år')
ax.set_xlabel('Pasientalder')
ax.set_ylabel('Tetthet')
ax.set_title('Distribusjonsskifte: når virkeligheten ser annerledes ut enn treningsdataene')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print('Hvis inn-data endrer seg, kan modellens ytelse falle uten at det oppdages med én gang.')

## 3. Usikkerhet og når AI bør si «jeg vet ikke»

En tillitsverdig modell bør ikke bare gi en anbefaling. Den bør også kunne uttrykke hvor sikker eller usikker den er.

To nyttige begreper er:

- **Epistemisk usikkerhet**: modellen mangler kunnskap, for eksempel fordi tilstanden er sjelden eller dårlig representert i treningsdata
- **Aleatorisk usikkerhet**: det er støy eller uklarhet i selve dataene, for eksempel dårlig bildekvalitet

I klinisk praksis er det ofte tryggere at AI-systemet signaliserer usikkerhet og utløser menneskelig vurdering, enn at det opptrer selvsikkert når grunnlaget er svakt.

In [ ]:
@dataclass
class KliniskPrediksjon:
    beskrivelse: str
    prediksjon: float
    usikkerhet: float
    usikkerhetstype: str


def vurder_handling(prediksjon, usikkerhet, terskel=0.20):
    if usikkerhet > terskel:
        return 'MANUELL_VURDERING'
    if prediksjon > 0.80:
        return 'HØY_KONFIDENS_POSITIV'
    if prediksjon < 0.20:
        return 'HØY_KONFIDENS_NEGATIV'
    return 'MANUELL_VURDERING_ANBEFALES'


cases = [
    KliniskPrediksjon('Typisk pneumoni, godt representert i data', 0.85, 0.05, 'aleatorisk'),
    KliniskPrediksjon('Atypisk presentasjon, få lignende tilfeller', 0.65, 0.25, 'epistemisk'),
    KliniskPrediksjon('Sjelden autoimmun tilstand', 0.55, 0.40, 'epistemisk')
]

for i, case in enumerate(cases, start=1):
    handling = vurder_handling(case.prediksjon, case.usikkerhet)
    print(f"Case {i}: {case.beskrivelse}")
    print(f"  Prediksjon: {case.prediksjon:.0%}")
    print(f"  Usikkerhet: ±{case.usikkerhet:.0%} ({case.usikkerhetstype})")
    print(f"  Anbefalt handling: {handling}\n")

## 4. Human-in-the-loop, validering og monitorering

### Human-in-the-loop
I medisinsk AI bør mennesker normalt være inne i eller over beslutningsløkken. Det betyr at:

- AI gir støtte, ikke endelig ansvar
- klinikeren kan overstyre anbefalingen
- systemet bør være laget for gjennomgang, ikke blind tillit

### Validering før innføring
Før et system tas i bruk, bør man minst spørre:

- Hvordan fungerte modellen på data den ikke ble trent på?
- Er den testet på andre pasientgrupper eller andre institusjoner?
- Finnes det subgrupper med tydelig svakere ytelse?

### Monitorering etter innføring
Selv et godt validert system kan forringes over tid. Derfor må man følge med på:

- endringer i input-data
- endringer i ytelse
- overridestatistikk fra klinikere
- antall usikre eller feilaktige anbefalinger

> En modell er ikke ferdig bare fordi den er deployet. Den må følges opp som annet medisinsk utstyr og klinisk beslutningsstøtte.

In [ ]:
def hitl_workflow(prediksjon, konfidens, kritisk=False):
    if kritisk and konfidens >= 0.95:
        return 'Haster: senior kliniker må varsles og bekrefte'
    if konfidens >= 0.80:
        return 'Vis anbefaling til kliniker for bekreftelse eller overstyring'
    return 'Flagg for manuell vurdering; AI brukes kun som støtteinformasjon'

scenarier = [
    ('Høy risiko for DVT', 0.92, True),
    ('Mulig pneumoni', 0.65, False),
    ('Høy risiko for lungeemboli', 0.97, True)
]

for navn, konfidens, kritisk in scenarier:
    print(f"Funn: {navn}")
    print(f"Konfidens: {konfidens:.0%}")
    print(f"Workflow: {hitl_workflow(navn, konfidens, kritisk)}\n")

## 5. Oppsummering

Et AI-system i helse er først virkelig tillitsverdig når det ikke bare leverer gode resultater i utviklingsfasen, men også:

- tåler endringer i data og kontekst
- signaliserer usikkerhet når det er grunn til det
- brukes med tydelig menneskelig kontroll
- er validert før innføring
- overvåkes etter at det er tatt i bruk

Denne notebooken henger tett sammen med både uke 06 og resten av uke 08:

- fra **uke 06** henter vi behovet for validering, generalisering og klinisk arbeidsflyt
- fra **uke 08** henter vi etikk, personvern, bias og regulering

Til sammen gir dette et mer realistisk bilde av hva som må til for å bruke AI forsvarlig i helse.